# hb1603_dive_analysis

Pooled HDBSCAN clustering of 13 HB1603 sperm whale dives with per-dive SVCodes, seabed masked on Sv.

**Author:** <Your Name>
**Version:** 1.0

In [ ]:
# Uncomment the lines below to install required packages.
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Utils.git
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_calibration.git
# %pip install echopype>=0.6.0
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_ML.git

In [ ]:
from aa_si_utils.dive_profiles import plan_dive_datasets
from aa_si_utils.data_retrieval import query_ncei_data
from aa_si_calibration.calibration import read_raw_file_config
from aa_si_calibration.calibration import record_raw_file_configs
from aa_si_calibration.calibration import standardize_calibration_files
from aa_si_calibration.calibration import build_calibration_mapping
from aa_si_utils.utils import read_raw_files_to_stores
from aa_si_utils.utils import combine_raw_stores
from aa_si_calibration.calibration import extract_standardized_calibration_parameters
from echopype.calibrate import compute_Sv
from aa_si_utils.utils import compute_transducer_depth
from echopype.consolidate import add_depth
from aa_si_utils.utils import crop_range_samples
from aa_si_utils.utils import select_ping_time_range
from aa_si_utils.utils import create_surface_mask
from aa_si_utils.utils import create_frequency_mask
from aa_si_utils.utils import read_seafloor_line_evl
from aa_si_utils.utils import create_seafloor_mask
from aa_si_utils.utils import combine_masks
from aa_si_utils.utils import apply_mask_to_sv
from aa_si_ml.ml import remove_noise
from aa_si_utils.utils import mask_sparse_bins
from echopype.commongrid import compute_MVBS
from aa_si_ml.ml import compute_per_cell_statistics
from aa_si_utils.utils import concat_datasets
from aa_si_utils.utils import rechunk_dataset
from aa_si_ml.ml import reshape_data_for_ml
from aa_si_ml.ml import add_auxiliary_features
from aa_si_ml.ml import normalize_data
from aa_si_ml.ml import run_hdbscan
from aa_si_ml.ml import assign_clusters_by_prediction
from aa_si_ml.ml import embed_clustering_results
from aa_si_ml.ml import plot_clustering_report
from aa_si_utils.dive_profiles import generate_sv_codes
from aa_si_utils.dive_profiles import sv_code_depth_table


In [ ]:
cal_input_folder = '../calibration_files/HB201607_cal'  # path
calibration_output_dir = './outputs/calibration'  # path
crop_max_range_m = 1800.0  # float
ping_time_bin = '10s'  # str
cruise_name = 'HB1603'  # str
file_time_start = '2016-06-01T00:00:00'  # str
file_time_end = '2016-09-01T00:00:00'  # str
cruise_id = 'HB1603'  # str
record_author = '<Your Name>'  # str
max_scan_bytes = 1048576  # int
match_tolerances = {'transmit_power': 1001.0}  # dict
sonar_model = 'EK60'  # str

## Included: dive_plan.yaml

Steps: plan_dives

### Step: `plan_dives`
**Op:** `custom` (custom / unregistered)

Turn the NEFSC dive configs into recipe windows.

**Parameters:**
- `json_dir` path
- `evl_root` path
- `pad_minutes` float
- `include_labels` list

In [ ]:
# --- Parameters ---
_plan_dives__json_dir = './Auxiliary/JSON_Files'
_plan_dives__evl_root = './Auxiliary'
_plan_dives__pad_minutes = 0.0
_plan_dives__include_labels = ['SWD_20160707-OE20', 'SWD_20160707-OE29', 'SWD_20160707-OE32', 'SWD_20160719-OE07', 'SWD_20160725-OE50', 'SWD_20160725-OE54', 'SWD_20160725-OE61', 'SWD_20160725-OE62', 'SWD_20160725-OE64', 'SWD_20160802-OE16', 'SWD_20160814-OE03', 'SWD_20160814-OE09', 'SWD_20160814-OE44']

_result = plan_dive_datasets(
    json_dir=_plan_dives__json_dir,
    evl_root=_plan_dives__evl_root,
    pad_minutes=_plan_dives__pad_minutes,
    include_labels=_plan_dives__include_labels,
)
windows = _result['windows']
raw_files = _result['raw_files']

*End of included section: dive_plan.yaml*

## Included: processing_lvl_0_user.yaml

Steps: query_ncei, scan_raw_config, record_raw_configs, standardize_cal, build_cal_mapping

### Step: `query_ncei`
**Op:** `query_ncei_data`

Query an NCEI acoustic data catalog for file listings within a time window. Returns a list of result records (file metadata), not the files themselves.

**Parameters:**
- `file_time_start` str - ISO 8601 start of window, e.g. '2016-07-25T20:58'
- `file_time_end` str - ISO 8601 end of window
- `filters` dict - Key-value filters applied to NCEI metadata (e.g. CRUISE_NAME)
- `file_names` list - Keep only these files, matched on the D{date}-T{time} stamp in the name. A name the query did not return is an error. Empty keeps everything.


In [ ]:
# --- Parameters ---
_query_ncei__file_time_start = '2016-06-01T00:00:00'
_query_ncei__file_time_end = '2016-09-01T00:00:00'

_result = query_ncei_data(
    file_time_start=_query_ncei__file_time_start,
    file_time_end=_query_ncei__file_time_end,
    filters={'CRUISE_NAME': 'HB1603'},
)
ncei_results = _result['records']
query_label = _result['query_label']
raw_urls = _result['raw_urls']

### Step: `scan_raw_config`
**Op:** `read_raw_file_config`

Read one raw file's channel configuration from its datagram headers. Map it over the raw file paths and collect the results with record_raw_file_configs.

**Parameters:**
- `raw_file_path` path - One .raw file, as a local path or a remote URL. Normally wired to ${_item}.

- `verify_start_time` bool - If true, also read EK80 files in full to verify metadata_start_time, which is slower. The default single pass scan is enough for channel configuration.

- `verbose` bool - Print per-file progress. Set false on a long fan-out to keep the log readable.

- `max_scan_bytes` int - Most leading bytes of a remote EK60 or EK80 file to read instead of the whole file, in bytes. Unset reads whole files. Around 1 MiB suits EK60; FM EK80 needs tens of MiB. Too low only costs a whole-file read.


*Runs once per `map_over` instance (fan-out loop).*

In [ ]:
file_config = []
for _item in raw_urls:
    _iter_file_config = read_raw_file_config(
        raw_file_path=_item,
        verbose=False,
        max_scan_bytes=1048576,
    )
    file_config.append(_iter_file_config)

### Step: `record_raw_configs`
**Op:** `record_raw_file_configs`

Collect per-file channel configurations into raw_file_configs/raw_file_configs.yaml under the calibration output directory. Use collect on the scan step's file_config output.

**Parameters:**
- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `verbose` bool - Print a summary of what was recorded.

In [ ]:
# --- Parameters ---
_record_raw_configs__output_base = './outputs/calibration'

_result = record_raw_file_configs(file_configs=file_config, output_base=_record_raw_configs__output_base)
raw_file_configs = _result['raw_file_configs']
frequencies = _result['frequencies']
raw_configs_path = _result['raw_configs_path']

### Step: `standardize_cal`
**Op:** `standardize_calibration_files`

Convert manufacturer calibration files (EK60 .cal or EK80 .xml) to standardized single-channel .yaml files under single_channel_calibration_files/. Skipped when the manufacturer files are unchanged.

**Parameters:**
- `cal_input_folder` path - Folder of manufacturer calibration files (.cal for EK60, .xml for EK80), local or a remote URL.

- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `cruise_id` str - Cruise identifier recorded in every generated single-channel file (e.g. "HB2407"), so a standardized file stays traceable to the survey it came from.

- `record_author` str - Name recorded as the author of every generated single-channel file. Not used in any calculation.

- `short_filenames` bool - Use compact names for the generated single-channel files. Set false for longer, more descriptive names.

- `overwrite` bool - Re-parse and rewrite even when the sidecar says the existing files are current, forcing a clean rebuild of the single-channel folder.

- `verbose` bool - Print the parsed parameters and the files written.
- `override_channels` dict - Write these channels as the single-channel files instead of parsing the manufacturer files. Same shape as the single_channel_data output: omit an entry to discard that channel, and edit values in place to correct them.


In [ ]:
# --- Parameters ---
_standardize_cal__cal_input_folder = '../calibration_files/HB201607_cal'
_standardize_cal__output_base = './outputs/calibration'
_standardize_cal__cruise_id = 'HB1603'
_standardize_cal__record_author = '<Your Name>'
_standardize_cal__short_filenames = True

_result = standardize_calibration_files(
    frequencies=frequencies,
    cal_input_folder=_standardize_cal__cal_input_folder,
    output_base=_standardize_cal__output_base,
    cruise_id=_standardize_cal__cruise_id,
    record_author=_standardize_cal__record_author,
    short_filenames=_standardize_cal__short_filenames,
)
single_channel_dir = _result['single_channel_dir']
channel_count = _result['channel_count']
skipped = _result['skipped']
standardize_cal_single_channel_data = _result['single_channel_data']

### Step: `build_cal_mapping`
**Op:** `build_calibration_mapping`

Match each raw channel to its calibration data and save the mapping files under mapping_files/. Mark the step regenerate: always in a recipe.

**Parameters:**
- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `single_channel_dir` path - Folder of standardized single-channel files. Defaults to single_channel_calibration_files/ under output_base. Wire it to the standardization step's single_channel_dir output.

- `raw_configs_path` path - Path of the saved raw_file_configs.yaml. Defaults to raw_file_configs/raw_file_configs.yaml under output_base, and is only consulted when raw_file_configs is not supplied.

- `conflict_resolution` str - What to do when one raw channel matches several calibration files. "error" (default) lists the conflicts and stops. "interactive" prompts on the terminal. "report" returns them on the conflicts output and writes no mapping.

- `calibration_choices` dict - Maps a conflict id from a previous run's conflicts output to a choice: the key to keep; a list of keys, or {"average": [keys]}, to average; or {"merge": [items]} to merge FM calibrations EK80 style, an item being a key or an average.

- `tolerances` dict - Per-field match tolerances, merged over the defaults (1 Hz for the frequencies, 1 W for transmit_power, 1e-6 s for transmit_duration_nominal). Widening one applies a calibration measured at a different setting.

- `unmapped_channels` str - The unmapped_channels policy, "warn" or "error", of the step that extracts calibration parameters from this mapping. Only labels the provenance report; unset says the outcome is decided downstream.

- `cruise_id` str - Recorded in the provenance report when given.
- `record_author` str - Recorded as the author of any record averaged or merged while resolving conflicts. When omitted, the combined calibrations' author is kept if they share one.

- `keep_unused` bool - Move calibration files no raw channel matched into unused_calibration_files/ instead of deleting them. They are the evidence when a channel comes out unmapped.

- `short_filenames` bool - Use compact calibration keys in the returned dictionaries and for averaged and merged records. Match the setting the single-channel files were written with.

- `verbose` bool - Print the mapping summary, preview, and verification results.

In [ ]:
# --- Parameters ---
_build_cal_mapping__output_base = './outputs/calibration'
_build_cal_mapping__single_channel_dir = single_channel_dir
_build_cal_mapping__raw_configs_path = raw_configs_path
_build_cal_mapping__short_filenames = True
_build_cal_mapping__keep_unused = True
_build_cal_mapping__conflict_resolution = 'error'

_result = build_calibration_mapping(
    raw_file_configs=raw_file_configs,
    output_base=_build_cal_mapping__output_base,
    single_channel_dir=_build_cal_mapping__single_channel_dir,
    raw_configs_path=_build_cal_mapping__raw_configs_path,
    short_filenames=_build_cal_mapping__short_filenames,
    tolerances={'transmit_power': 1001.0},
    keep_unused=_build_cal_mapping__keep_unused,
    conflict_resolution=_build_cal_mapping__conflict_resolution,
)
mapping_dict = _result['mapping_dict']
calibration_dict = _result['calibration_dict']
missing_params = _result['missing_params']
unused_files = _result['unused_file_names']
provenance = _result['provenance']
provenance_path = _result['provenance_path']
conflicts = _result['conflicts']
build_cal_mapping_single_channel_data = _result['single_channel_data']

*End of included section: processing_lvl_0_user.yaml*

## Included: processing_lvl_1_per_file.yaml

Steps: read_raw, combine_raw

### Step: `read_raw`
**Op:** `read_raw_files`

Open raw echosounder files with echopype and write each one to an intermediate store. Returns the store paths, or in-memory EchoData objects when intermediate_format is "none".

**Parameters:**
- `sonar_model` str - Sonar model string recognized by echopype (e.g. 'EK60', 'EK80')
- `include_bot` bool - Include bottom-detection (.bot) data if available
- `intermediate_format` str - Store format written between reading and combining. "netcdf" (default, recommended) writes temporary .nc files. "none" keeps everything in memory. "zarr" writes zarr stores, which can live on a gs:// temp dir.

- `use_swap` str - Whether echopype swaps data to disk while reading. "auto" (default) swaps a file whose expanded data would exceed about 40% of total RAM.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `combine_raw`
**Op:** `combine_raw_files`

Combine the per-file stores or EchoData objects from read_raw_files into one EchoData object. Mark the step checkpoint: always so the combined output is written once and reused.

**Parameters:**
- `ping_time_chunk` int - Chunk length along ping_time for the combined EchoData and its checkpoint, in pings. Larger values write faster to gs:// but use more memory; try 2000 to 4000 for bucket caches.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `extract_cal_params`
**Op:** `extract_standardized_cal_params`

Extract calibration parameters (gain, sa-correction, EBA) and environmental parameters (sound speed, absorption) from standardized calibration files, cross-referenced against the EchoData object for channel alignment.

**Parameters:**
- `raw_file_path` path - Raw file whose mapping to use, matched by basename. Wire to ${_item} in a per-file chain. Unset uses the first file in the survey, which suits whole-survey recipes.

- `unmapped_channels` str - What to do with a channel no calibration file matched. "warn" (default) reports it and uses the values the raw file records. "error" stops the run.

- `output_logs_folder` path - Where the fallback writes its calibration flags. Unset uses a temporary folder.

- `waveform_mode` str - The waveform_mode compute_sv uses, "CW" or "BB". Selects which beam group's channels are extracted, for EK80 files mixing CW and FM. Unset uses the first beam group.

- `encode_mode` str - The encode_mode the compute_sv step passes to echopype, "complex" or "power". Set it together with waveform_mode, or leave both unset.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `compute_sv`
**Op:** `compute_sv`

Compute volume backscattering strength (Sv) from raw EchoData, applying cal_params and env_params when given and the raw file's own values otherwise.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `compute_transducer_depth`
**Op:** `compute_transducer_depth`

Compute per-ping transducer depth in metres below the water surface from the EK Platform group, for feeding ``ep_add_depth`` as its ``depth_offset``. Use it instead of ``use_platform_vertical_offsets``, which gets the sign wrong.

**Parameters:**
- `datum_correction_m` float (m) - Metres added to the static offset when it is measured from a hull datum rather than the waterline. Positive pushes the transducer deeper. The default 0.0 takes the recorded offset as waterline referenced.

- `use_heave` bool - Add the per-ping ``vertical_offset`` (heave) term. False holds the static keel depth on every ping. Without usable heave the step says so and uses the static depth.

- `heave_sign` float - Sign applied to ``vertical_offset`` (1.0 or -1.0). The default was verified on HB2407 (EK80); check it on a new vessel or instrument setup.

- `interp_method` str - Interpolation used to put heave on ping_time, such as "linear" (default) or "nearest".


*Runs once per `map_over` instance (fan-out loop).*

### Step: `ep_add_depth`
**Op:** `ep_add_depth`

Add a ``depth`` variable, in metres below the surface, to an Sv Dataset using echopype. Existing variables, including ``echo_range``, are kept.

**Parameters:**
- `depth_offset` (m) - Transducer depth in metres, as a constant or a per-ping DataArray such as ``compute_transducer_depth``'s output. When set, takes precedence over Platform vertical offsets.

- `tilt` float (degree) - Constant transducer tilt; 0 corresponds to a vertically-pointing transducer. When set, takes precedence over Platform/Beam angles.

- `downward` bool - True when transducers point downward (typical hull-mounted).
- `use_platform_vertical_offsets` bool - Use Echodata Platform group vertical offsets to compute transducer depth. EK60/EK80 only. Ignored when ``depth_offset`` is set.

- `use_platform_angles` bool - Use Echodata Platform group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_beam_angles``.

- `use_beam_angles` bool - Use Echodata Beam group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_platform_angles``.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `crop_survey_range`
**Op:** `crop_range`

Trim the deep range samples that hold no data after seafloor masking. Run it on the whole survey, after the per-file fan-in. Active criteria each give a cut and the shallowest wins.

**Parameters:**
- `max_range_m` float (m) - Hard cap on range. Samples deeper than this are dropped whether or not they hold data. Unset derives the crop from the data.

- `outlier_sigma` float - Standard deviations above the mean of each ping's deepest finite sample at which a ping counts as an outlier and is ignored. Unset disables outlier rejection and uses drop_trailing_all_nan.

- `outlier_margin` float - Multiplier on (mean + outlier_sigma x sd) giving the outlier threshold. Raise it to flag fewer pings. Only used when outlier_sigma is set.

- `drop_trailing_all_nan` bool - Crop to one past the deepest sample with a finite value anywhere (lossless). Ignored when outlier_sigma is set. False, with the other two unset, turns the step into a no-op.

- `data_var` str - Variable inspected for finite values.
- `range_var` str - Variable supplying each sample's range in metres.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_sv`
**Op:** `select_ping_time_range`

Narrow a Dataset to one or more ping_time windows, for example to work on a segment of a shared survey-wide Sv store. Keep the window on this step, not on upstream steps, so their cache stays shared.

**Parameters:**
- `start` str - Inclusive ISO datetime lower bound (e.g. "2024-10-15T13:38:52"). Leave unset to leave the start open.

- `end` str - Inclusive ISO datetime upper bound. Leave unset to leave the end open.

- `window` dict - Dict with ``start`` and ``end``, used instead of those params; wire as ``window: ${_item}`` when mapping over windows. Other keys are ignored and an optional ``label`` is logged.

- `dim` str - Datetime coordinate to slice. Use "cell_ping_time" for a compute_per_cell_statistics dataset; raw Sv uses the default.

- `windows` list - Several window dicts, each with ``start`` and ``end``. Keeps every ping inside at least one, in time order, once each. Often wired from a planner with ``allow_empty: true``.

- `snap_to_bin` str - Bin width such as "10s". Widens every window outward to whole bins of that width. Set it to the downstream ping_time_bin when this feeds compute_mvbs. Unset selects the exact bounds.

- `allow_empty` bool - Return nothing instead of failing when the window selects no pings. Set true when mapping over per-file Sv; empty instances drop out at the fan-in.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_surface_mask`
**Op:** `create_surface_mask`

Build a surface-interference mask from baseline Sv using a selectable method.

**Parameters:**
- `surface_depth_m` float (m) - Depth above which samples are masked out, removing the transmit pulse ringdown and near-surface bubble layer. The default of 0.0 masks nothing.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_frequency_mask`
**Op:** `create_frequency_mask`

Build a mask that excludes selected frequency channels. Implementation can be swapped to use different exclusion criteria.

**Parameters:**
- `frequencies_to_mask` list (kHz) - Nominal channel frequencies to exclude, in kHz (e.g. [70, 120, 200]). Every sample on a listed channel is marked invalid. The default empty list masks nothing.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_seabed_line`
**Op:** `read_seafloor_line`

Read a seafloor line from Echoview ``.evl`` files and align it to the Sv pings, in metres. Swaps in for the seafloor detection ops. Pings the line does not cover are NaN and get masked entirely downstream.

**Parameters:**
- `evl_path` path - An Echoview ``.evl`` line file, or a folder of them selected with file_time_start and file_time_end. Local or remote (e.g. gs://...). Give exactly one of this and ``window``.

- `window` dict - Mapping with the line path under ``window_key``, used instead of ``evl_path`` in a mapped step (``window: ${_item}``). It can also supply file_time_start and file_time_end.

- `window_key` str - Key the line path is read from when ``window`` is given. Defaults to "seabed_evl", which is what plan_dive_datasets emits.

- `vertical_reference` str - What the EVL depths are measured from: "surface" (default, Echoview's usual) or "transducer" (range along the beam). The line is converted to ds_Sv's own reference.

- `min_status` int - Drop line points whose Echoview status is below this value (0 = none, 1 = unverified, 2 = bad, 3 = good). Null (default) keeps every point. Dropped points leave a gap, subject to max_gap_s.

- `max_gap_s` float (s) - Widest hole in the line to interpolate across; pings in a wider hole are NaN. Null (default) interpolates across holes of any width.

- `edge_extend_s` float (s) - Seconds past the line's first and last point to hold its end depths. 0.0 (default) leaves pings outside the line NaN; null holds the end depths indefinitely.

- `depth_offset_m` float (m) - Constant metres added to the line, for when the transducer draft configured in Echoview differs from the one baked into ds_Sv['depth']. Positive pushes the seafloor deeper.

- `min_coverage` float - Fraction of pings (0-1) that must end up with a finite seafloor depth; below it the step fails instead of silently masking the uncovered pings away. Default 0.0 never fails.

- `file_time_start` str - Optional inclusive ISO datetime (e.g. "2024-10-12T00:15") selecting line files in an evl_path folder by the times in their names. Use the window that selected the raw files. Ignored for a single file.

- `file_time_end` str - Optional inclusive ISO datetime upper bound for line file selection. See file_time_start.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_seabed_mask`
**Op:** `create_seafloor_mask`

Build a seafloor mask from baseline Sv and detected seafloor depth. The seafloor depth is passed as an explicit input, enabling swappable detection methods without modifying masking logic.

**Parameters:**
- `seafloor_buffer_m` float (m) - Height above the seafloor line to also mask, covering detection error and the near-bottom dead zone at the cost of some real backscatter.

- `range_var` str - Meter-valued variable on ds_Sv compared against the seafloor line, on the same vertical reference. Null (default) uses "depth" if present, otherwise "echo_range".


*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_combine_masks`
**Op:** `combine_masks`

Combine two or more mask datasets into one final mask. Accepts a list of mask inputs to support inserting additional custom mask-producing steps into the chain without changing core framework code.

**Parameters:**
- `mode` str - Combination mode: 'and' (all must be valid) or 'or' (any must be valid)

*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_apply_mask`
**Op:** `apply_sv_mask`

Apply a boolean mask to an Sv Dataset, setting masked samples to NaN.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_remove_noise`
**Op:** `remove_background_noise`

Estimate and remove background noise from Sv using the De Robertis and Higginbottom method. Returns a cleaned Sv Dataset whose Sv holds the corrected values.

**Parameters:**
- `ping_num` int - Number of pings used to estimate the noise level
- `range_sample_num` int - Number of range samples used to estimate the noise level
- `background_noise_max` str - Upper limit on the estimated noise level (e.g. '-125dB'); omit to leave it unbounded
- `SNR_threshold` str - Signal-to-noise ratio below which corrected samples are masked out (e.g. '3.0dB')

*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_mask_sparse`
**Op:** `mask_sparse_bins`

Remove spatiotemporal bins with fewer valid samples than a threshold, reducing noise in MVBS computation.

**Parameters:**
- `range_bin` str - Range bin size (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size (e.g. '10s')
- `nan_threshold` float - Fraction of NaN values above which a bin is masked (0.0 to 1.0)
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth). Must match compute_mvbs and compute_per_cell_statistics.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_file_mvbs`
**Op:** `compute_mvbs`

Compute Mean Volume Backscattering Strength by averaging Sv over range and time bins. Returns a gridded MVBS Dataset.

**Parameters:**
- `range_bin` str (m) - Depth bin size for averaging (e.g. '2m')
- `ping_time_bin` str (s) - Ping-time bin size for averaging (e.g. '10s')
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth, which ep_add_depth adds).

- `range_var_max` str (m) - Deepest value the bins cover, as a unit string (e.g. '2100m'). Unset reads the maximum from the data; supplying it keeps the step lazy.

- `closed` str - Which side of each bin interval is closed: 'left' or 'right'
- `skipna` bool - Skip NaN values when averaging. When false, any NaN in a bin makes the whole bin NaN, so masked data collapses.

- `fill_value` float - Value for bins holding no data. Unset keeps echopype's default, NaN.

- `method` str - Flox reduction strategy for dask arrays, normally 'map-reduce'. Any other value needs reindex set to null in the same params block.

- `reindex` bool - Reindex at the blockwise stage rather than after it. False uses less memory and more time. Only meaningful when method is 'map-reduce'.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `dive_cell_stats`
**Op:** `compute_per_cell_statistics`

Compute statistics within each MVBS cell (such as the coefficient of variation) from fine-resolution Sv, at MVBS bin resolution. Use the same bins as mask_sparse_bins and compute_mvbs, and pass the result to add_auxiliary_features as ds_sv.

**Parameters:**
- `range_bin` str - Range bin size. Must match mask_sparse_bins and compute_mvbs (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size. Must match mask_sparse_bins and compute_mvbs (e.g. '10s')
- `statistics` list - List of statistics to compute. Supported: ['cv']
- `data_var` str - Name of the Sv variable in the input Dataset
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth). Must match mask_sparse_bins and compute_mvbs.

- `range_var_max` str (m) - Deepest value the cells cover, as a unit string (e.g. '1810m'), matching compute_mvbs. Unset reads the extent from the data. Pin it whenever the results will be merged.

- `cells_only` bool - Return only the statistic arrays instead of a copy of ds_Sv carrying them. Set true whenever this step is checkpointed or its results are merged.


*Runs once per `map_over` instance (fan-out loop).*

In [ ]:
raw_stores = []
echodata = []
cal_params = []
env_params = []
other_params = []
compute_sv_ds_Sv = []
transducer_depth = []
ep_add_depth_ds_Sv = []
crop_survey_range_ds_Sv = []
dive_sv_ds_Sv = []
dive_surface_mask_mask = []
dive_frequency_mask_mask = []
seafloor_depth = []
dive_seabed_mask_mask = []
dive_combine_masks_mask = []
dive_apply_mask_ds_Sv = []
dive_remove_noise_ds_Sv = []
dive_mask_sparse_ds_Sv = []
ds_MVBS = []
dive_cell_stats_ds_Sv = []
for _item in raw_urls:
    _iter_raw_stores = read_raw_files_to_stores(
        raw_file_paths=[_item],
        sonar_model='EK60',
        include_bot=False,
        intermediate_format='zarr',
    )
    raw_stores.append(_iter_raw_stores)
    _iter_echodata = combine_raw_stores(raw_stores=_iter_raw_stores)
    echodata.append(_iter_echodata)
    _result = extract_standardized_calibration_parameters(
        calibration_dict=calibration_dict,
        mapping_dict=mapping_dict,
        echodata=_iter_echodata,
        raw_file_path=_item,
    )
    _iter_cal_params = _result['cal_params']
    _iter_env_params = _result['env_params']
    _iter_other_params = _result['other_params']
    cal_params.append(_iter_cal_params)
    env_params.append(_iter_env_params)
    other_params.append(_iter_other_params)
    _iter_compute_sv_ds_Sv = compute_Sv(
        echodata=_iter_echodata,
        cal_params=_iter_cal_params,
        env_params=_iter_env_params,
    )
    compute_sv_ds_Sv.append(_iter_compute_sv_ds_Sv)
    _iter_transducer_depth = compute_transducer_depth(
        ds_Sv=_iter_compute_sv_ds_Sv,
        echodata=_iter_echodata,
        use_heave=True,
        heave_sign=1.0,
    )
    transducer_depth.append(_iter_transducer_depth)
    _iter_ep_add_depth_ds_Sv = add_depth(
        ds=_iter_compute_sv_ds_Sv,
        echodata=_iter_echodata,
        depth_offset=_iter_transducer_depth,
    )
    ep_add_depth_ds_Sv.append(_iter_ep_add_depth_ds_Sv)
    _iter_crop_survey_range_ds_Sv = crop_range_samples(
        ds_Sv=_iter_ep_add_depth_ds_Sv,
        max_range_m=1800.0,
        drop_trailing_all_nan=False,
        outlier_sigma=None,
    )
    crop_survey_range_ds_Sv.append(_iter_crop_survey_range_ds_Sv)
    _iter_dive_sv_ds_Sv = select_ping_time_range(
        ds_Sv=_iter_ep_add_depth_ds_Sv,
        windows=windows,
        allow_empty=True,
        snap_to_bin='10s',
    )
    dive_sv_ds_Sv.append(_iter_dive_sv_ds_Sv)
    _iter_dive_surface_mask_mask = create_surface_mask(ds_Sv=_iter_dive_sv_ds_Sv, surface_depth_m=10.0)
    dive_surface_mask_mask.append(_iter_dive_surface_mask_mask)
    _iter_dive_frequency_mask_mask = create_frequency_mask(ds_Sv=_iter_dive_sv_ds_Sv, frequencies_to_mask=[70, 120, 200])
    dive_frequency_mask_mask.append(_iter_dive_frequency_mask_mask)
    _iter_seafloor_depth = read_seafloor_line_evl(
        ds_Sv=_iter_dive_sv_ds_Sv,
        evl_path='./Auxiliary/seabed_lines',
        min_coverage=0.99,
    )
    seafloor_depth.append(_iter_seafloor_depth)
    _iter_dive_seabed_mask_mask = create_seafloor_mask(
        ds_Sv=_iter_dive_sv_ds_Sv,
        seafloor_depth=_iter_seafloor_depth,
        seafloor_buffer_m=10.0,
    )
    dive_seabed_mask_mask.append(_iter_dive_seabed_mask_mask)
    _iter_dive_combine_masks_mask = combine_masks(masks=[_iter_dive_surface_mask_mask, _iter_dive_frequency_mask_mask, _iter_dive_seabed_mask_mask], mode='and')
    dive_combine_masks_mask.append(_iter_dive_combine_masks_mask)
    _iter_dive_apply_mask_ds_Sv = apply_mask_to_sv(ds_Sv=_iter_dive_sv_ds_Sv, mask=_iter_dive_combine_masks_mask)
    dive_apply_mask_ds_Sv.append(_iter_dive_apply_mask_ds_Sv)
    _iter_dive_remove_noise_ds_Sv = remove_noise(
        ds_Sv=_iter_dive_apply_mask_ds_Sv,
        range_sample_num=10,
        ping_num=5,
        SNR_threshold='6.0dB',
        background_noise_max='-66dB',
    )
    dive_remove_noise_ds_Sv.append(_iter_dive_remove_noise_ds_Sv)
    _iter_dive_mask_sparse_ds_Sv = mask_sparse_bins(
        ds_Sv=_iter_dive_remove_noise_ds_Sv,
        range_bin='2m',
        ping_time_bin='10s',
        range_var='depth',
        nan_threshold=0.99,
    )
    dive_mask_sparse_ds_Sv.append(_iter_dive_mask_sparse_ds_Sv)
    _iter_ds_MVBS = compute_MVBS(
        ds_Sv=_iter_dive_mask_sparse_ds_Sv,
        range_bin='2m',
        ping_time_bin='10s',
        range_var='depth',
        range_var_max='3020m',
    )
    ds_MVBS.append(_iter_ds_MVBS)
    _iter_dive_cell_stats_ds_Sv = compute_per_cell_statistics(
        ds_Sv=_iter_dive_remove_noise_ds_Sv,
        range_bin='2m',
        ping_time_bin='10s',
        range_var='depth',
        range_var_max='3020m',
        statistics=['cv'],
        cells_only=True,
    )
    dive_cell_stats_ds_Sv.append(_iter_dive_cell_stats_ds_Sv)

*End of included section: processing_lvl_1_per_file.yaml*

## Included: processing_lvl_2_combined.yaml

Steps: extract_cal_params, compute_sv, compute_transducer_depth, ep_add_depth, crop_survey_range, dive_sv, dive_surface_mask, dive_frequency_mask, dive_seabed_line, dive_seabed_mask, dive_combine_masks, dive_apply_mask, dive_remove_noise

*End of included section: processing_lvl_2_combined.yaml*

## Included: processing_lvl_3_user.yaml

Steps: dive_mask_sparse, dive_file_mvbs, dive_cell_stats, merge_dive_files_mvbs, rechunk_dive_mvbs, merge_dive_files_cell_stats

### Step: `merge_dive_files_mvbs`
**Op:** `merge_datasets`

Concatenate a collected list of Datasets into one along ``dim``, for example per-file MVBS gathered with ``collect``. A single Dataset passes through unchanged.

**Parameters:**
- `dim` str - Dimension to concatenate the collected Datasets along.
- `on_duplicate` str - How to handle two collected segments with the same label on ``dim``. Unset keeps both (fine when segments can't overlap). "mean" averages them (binned products split at a segment boundary). "first" keeps the first and sorts (overlapping slices of one store).


In [ ]:
# --- Parameters ---
_merge_dive_files_mvbs__dim = 'ping_time'
_merge_dive_files_mvbs__on_duplicate = 'mean'

merge_dive_files_mvbs_merged = concat_datasets(
    datasets=ds_MVBS,
    dim=_merge_dive_files_mvbs__dim,
    on_duplicate=_merge_dive_files_mvbs__on_duplicate,
)

### Step: `rechunk_dive_mvbs`
**Op:** `rechunk_dataset`

Rechunk a Dataset along ping_time so the steps after it can run dask-parallel.

**Parameters:**
- `ping_time_chunk` int - Target chunk size along ping_time.

In [ ]:
# --- Parameters ---
_rechunk_dive_mvbs__ping_time_chunk = 1000

rechunk_dive_mvbs_ds_Sv = rechunk_dataset(ds_Sv=merge_dive_files_mvbs_merged, ping_time_chunk=_rechunk_dive_mvbs__ping_time_chunk)

### Step: `merge_dive_files_cell_stats`
**Op:** `merge_datasets`

Concatenate a collected list of Datasets into one along ``dim``, for example per-file MVBS gathered with ``collect``. A single Dataset passes through unchanged.

**Parameters:**
- `dim` str - Dimension to concatenate the collected Datasets along.
- `on_duplicate` str - How to handle two collected segments with the same label on ``dim``. Unset keeps both (fine when segments can't overlap). "mean" averages them (binned products split at a segment boundary). "first" keeps the first and sorts (overlapping slices of one store).


In [ ]:
# --- Parameters ---
_merge_dive_files_cell_stats__dim = 'cell_ping_time'
_merge_dive_files_cell_stats__on_duplicate = 'mean'

merge_dive_files_cell_stats_merged = concat_datasets(
    datasets=dive_cell_stats_ds_Sv,
    dim=_merge_dive_files_cell_stats__dim,
    on_duplicate=_merge_dive_files_cell_stats__on_duplicate,
)

*End of included section: processing_lvl_3_user.yaml*

## Included: processing_lvl_4.yaml

Steps: reshape_ml, add_aux_features, normalize_ml, run_hdbscan, label_all_points, embed_results, plot_clustering_report

### Step: `reshape_ml`
**Op:** `reshape_for_ml`

Reshape an MVBS Dataset into a flat, ML-ready xarray Dataset where each observation corresponds to one spatial or temporal bin.

**Parameters:**
- `data_var` str - Name of the data variable to use as features
- `dataset_name` str - Name this ML dataset is stored under inside the returned Dataset. Every later ML step takes the same dataset_name to find it, so a recipe can carry several ML datasets side by side.

- `feature_strategy` str - Feature construction strategy (e.g. 'mean_centered')
- `baseline_channel` int - Channel index used as the baseline when computing relative features

In [ ]:
# --- Parameters ---
_reshape_ml__data_var = 'Sv'
_reshape_ml__dataset_name = 'ml_dataset'
_reshape_ml__feature_strategy = 'mean_centered'
_reshape_ml__baseline_channel = 0

reshape_ml_ds_ml_ready = reshape_data_for_ml(
    ds_Sv=rechunk_dive_mvbs_ds_Sv,
    data_var=_reshape_ml__data_var,
    dataset_name=_reshape_ml__dataset_name,
    feature_strategy=_reshape_ml__feature_strategy,
    baseline_channel=_reshape_ml__baseline_channel,
)

### Step: `add_aux_features`
**Op:** `add_auxiliary_features`

Append auxiliary coordinate-derived features (e.g. depth, ping_time_seconds) to the flattened ML-ready Dataset. Optional step; omit when no auxiliary features are needed.

**Parameters:**
- `dataset_name` str - ML dataset to append the features to. Must match the dataset_name used by reshape_for_ml.

- `features` list - List of built-in feature name strings to append. Supported built-ins: 'depth', 'ping_time_seconds', 'seafloor_depth', 'altitude', 'cell_cv'. 'seafloor_depth' and 'altitude' require the seafloor_depth or echodata input. 'cell_cv' requires the ds_sv input (from compute_per_cell_statistics).


In [ ]:
# --- Parameters ---
_add_aux_features__dataset_name = 'ml_dataset'
_add_aux_features__features = ['cell_cv']

add_aux_features_ds_ml_ready = add_auxiliary_features(
    ds_ml_ready=reshape_ml_ds_ml_ready,
    ds_sv=merge_dive_files_cell_stats_merged,
    dataset_name=_add_aux_features__dataset_name,
    features=_add_aux_features__features,
)

### Step: `normalize_ml`
**Op:** `normalize_ml_data`

Normalize ML features in the flattened Dataset. Supports per-feature normalization methods via per_group_methods.

**Parameters:**
- `method` str - Global normalization method (e.g. 'flatten', 'minmax', 'zscore')
- `dataset_name` str - ML dataset to normalize. Must match the dataset_name used by reshape_for_ml.

- `normalization_name` str - Name this normalized result is stored under. run_hdbscan, the ML plots, and filter_normalized_by_cluster take the same name to find it, so several normalizations of one dataset can coexist and be compared.

- `shift_positive` bool - Shift all values to be non-negative after normalization
- `feature_weights` list - Per-feature weight multipliers applied before normalization
- `per_group_methods` dict - Map of feature name to individual normalization method override

In [ ]:
# --- Parameters ---
_normalize_ml__method = 'flatten'
_normalize_ml__dataset_name = 'ml_dataset'
_normalize_ml__normalization_name = 'normalized_flatten_mean_centered'
_normalize_ml__shift_positive = False

normalize_ml_ds_normalized = normalize_data(
    ds_ml_ready=add_aux_features_ds_ml_ready,
    method=_normalize_ml__method,
    dataset_name=_normalize_ml__dataset_name,
    normalization_name=_normalize_ml__normalization_name,
    shift_positive=_normalize_ml__shift_positive,
)

### Step: `run_hdbscan`
**Op:** `run_hdbscan`

Run HDBSCAN unsupervised clustering on normalized feature data. A good sweep target for comparing hyperparameters.

**Parameters:**
- `dataset_name` str - ML dataset to cluster. Must match the dataset_name used by reshape_for_ml.

- `normalization_name` str - Normalized feature set to cluster. Must match the normalization_name used by normalize_ml_data.

- `ml_result_name` str - Name this clustering result is stored under. embed_clustering_results and plot_clustering_report take the same name to find it, so a recipe can keep several clusterings of one dataset side by side.

- `min_cluster_size` int - Absolute minimum cluster size. Mutually exclusive with min_cluster_size_fraction.
- `min_cluster_size_fraction` float - Minimum cluster size as a fraction of the points fitted, used when min_cluster_size is omitted. Mutually exclusive with min_cluster_size. Unset uses 0.03.

- `min_samples` int - Neighbourhood size HDBSCAN uses to judge how dense a point is. Higher values make the run more conservative, pushing sparse points into noise instead of into a cluster.

- `sample_size` int - Cap on how many samples are clustered. A larger dataset is clustered on a random subset of this size and the rest get no label (see assign_clusters_to_all_points). Set it above the sample count to cluster everything.

- `cluster_selection_method` str - HDBSCAN cluster selection method ('eom' or 'leaf')
- `use_hdbscan` bool - Use HDBSCAN; set false to fall back to a simpler clustering method
- `epsilon` float - DBSCAN epsilon distance parameter
- `find_background_cluster` bool - Run background-cluster detection path instead of standard HDBSCAN. When true, background_label output is populated.

- `soft_membership_threshold` float - Reassign noise points via soft-membership probabilities when set. null disables soft-membership reassignment.


In [ ]:
# --- Parameters ---
_run_hdbscan__dataset_name = 'ml_dataset'
_run_hdbscan__normalization_name = 'normalized_flatten_mean_centered'
_run_hdbscan__ml_result_name = 'hdbscan_results'
_run_hdbscan__min_cluster_size_fraction = 0.01
_run_hdbscan__min_samples = 4
_run_hdbscan__sample_size = 1200000
_run_hdbscan__cluster_selection_method = 'leaf'
_run_hdbscan__use_hdbscan = True
_run_hdbscan__soft_membership_threshold = 0.1

_result = run_hdbscan(
    ds_normalized=normalize_ml_ds_normalized,
    dataset_name=_run_hdbscan__dataset_name,
    normalization_name=_run_hdbscan__normalization_name,
    ml_result_name=_run_hdbscan__ml_result_name,
    min_cluster_size_fraction=_run_hdbscan__min_cluster_size_fraction,
    min_samples=_run_hdbscan__min_samples,
    sample_size=_run_hdbscan__sample_size,
    cluster_selection_method=_run_hdbscan__cluster_selection_method,
    use_hdbscan=_run_hdbscan__use_hdbscan,
    soft_membership_threshold=_run_hdbscan__soft_membership_threshold,
)
run_hdbscan_clustering_results = _result['clustering_results']
run_hdbscan_clustering_model = _result['clustering_model']
background_label = _result['background_label']
run_hdbscan_cluster_labels = _result['cluster_labels']

### Step: `label_all_points`
**Op:** `assign_clusters_to_all_points`

Label every point of a subsampled clustering, so no cell is left unlabelled. Points outside the fit get the cluster the fitted model predicts, or -1 for noise. Place it between ``run_hdbscan`` and ``embed_clustering_results``.

**Parameters:**
- `dataset_name` str - ML dataset the clustering belongs to. Must match the dataset_name used by reshape_for_ml and run_hdbscan.

- `normalization_name` str - Normalized feature set that was clustered. Must match the normalization_name used by normalize_ml_data and run_hdbscan.

- `ml_result_name` str - Name the extended result is stored under. Unset keeps the name already on the incoming result, so downstream embedding and plotting steps need no change.

- `batch_size` int - Rows predicted per batch. Lower it if a survey-scale run runs out of memory; batching does not change the labels.

- `keep_model` bool - Pass the fitted model through on the clustering_model output. Set false when this step is checkpointed and pickle is unwanted.


In [ ]:
# --- Parameters ---
_label_all_points__dataset_name = 'ml_dataset'
_label_all_points__normalization_name = 'normalized_flatten_mean_centered'

_result = assign_clusters_by_prediction(
    ds_normalized=normalize_ml_ds_normalized,
    clustering_results=run_hdbscan_clustering_results,
    clustering_model=run_hdbscan_clustering_model,
    dataset_name=_label_all_points__dataset_name,
    normalization_name=_label_all_points__normalization_name,
)
label_all_points_clustering_results = _result['clustering_results']
label_all_points_clustering_model = _result['clustering_model']
label_all_points_cluster_labels = _result['cluster_labels']

### Step: `embed_results`
**Op:** `embed_clustering_results`

Embed clustering results into the normalized ML Dataset as cluster-label variables. A collected list of results embeds one variable per result, named from ml_result_name.

**Parameters:**
- `dataset_name` str - ML dataset the labels are embedded into. Must match the dataset_name used by reshape_for_ml.

- `ml_result_name` str - Name the embedded cluster labels are stored under, normally the ml_result_name from run_hdbscan so plot_clustering_report finds it. For a collected list, the stem of the per-result variable names.


In [ ]:
# --- Parameters ---
_embed_results__dataset_name = 'ml_dataset'
_embed_results__ml_result_name = 'hdbscan_results'

_result = embed_clustering_results(
    ds_normalized=normalize_ml_ds_normalized,
    clustering_results=label_all_points_clustering_results,
    dataset_name=_embed_results__dataset_name,
    ml_result_name=_embed_results__ml_result_name,
)
embed_results_ds_normalized = _result['ds_normalized']
gridded_results = _result['gridded_results']

### Step: `plot_clustering_report`
**Op:** `plot_clustering_report`

Render a clustering report: the clustering echogram, per-cluster statistics, and the HDBSCAN hierarchy plot when a fitted model is available. Produces figures only.

**Parameters:**
- `dataset_name` str - ML dataset to report on. Must match the dataset_name used by reshape_for_ml.

- `ml_result_name` str - Clustering result to report on. Must match the ml_result_name used by embed_clustering_results, which is the name the labels were stored under in ds_normalized.

- `plot_window` list - [min_depth, max_depth, ping_min, ping_max] for echogram display
- `time_min` str - Start of the displayed window as a UTC timestamp, for example "2024-10-15T12:30". Resolved to the closest ping and used instead of the ping_min entry in plot_window. Omit to window by ping index.

- `time_max` str - End of the displayed window as a UTC timestamp. Resolved to the closest ping and used instead of the ping_max entry in plot_window. Omit to window by ping index.

- `x_axis_units` str - Units for the clustering echogram x-axis: 'datetime' (default, absolute UTC clock times), 'seconds' from the first ping shown, 'pings', 'bins', or 'meters'.

- `y_axis_units` str - Units for the clustering echogram y-axis ('meters', 'range_sample', or 'bins')
- `overlay_line_var` str - Name of a variable in ds_normalized to draw as a line overlay
- `overlay_windows` list - Window dicts (as plan_dive_datasets emits) whose line files are drawn onto the echogram. Empty draws nothing.

- `overlay_window_keys` list - Window keys naming the line files to draw. Defaults to the three dive profile keys dive_fit_evl, dive_u99_evl and dive_l99_evl. A window missing a key is skipped for that key.

- `overlay_fit_style` dict - Matplotlib line keywords for the fitted dive profile, for example {color: "#23FFFF", linewidth: 3.0}, merged over the built-in solid cyan. Empty uses the built-in style.

- `overlay_bound_style` dict - The same for both 99% confidence bound lines, for example {color: "#D08BFF", linestyle: ":"}. Empty uses the built-in dotted violet, thinner than the fitted profile.

- `normalization_name` str - Normalized feature set to plot statistics for when cluster_stats_sv_data_var is "ml_features" and no clustering_results input is wired. Ignored otherwise.

- `cluster_colors` list - Hex color strings for each cluster label
- `y_to_x_aspect_ratio_override` float - Override the echogram aspect ratio; null uses the default
- `cluster_stats_sv_data_var` str - Sv variable name used when computing per-cluster statistics, or "ml_features" to plot the flattened ML features used for clustering
- `cluster_stats_compute_pairwise_differences` bool - Compute inter-channel diffs in cluster statistics, taken against the second-lowest-frequency channel

- `cluster_stats_dpi` int - DPI for the cluster statistics figure, which holds much smaller text than the echogram. Null falls back to dpi.

- `save_image` bool - Save the rendered figures to image files; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display figures interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_clustering_report__dataset_name = 'ml_dataset'
_plot_clustering_report__ml_result_name = 'hdbscan_results'
_plot_clustering_report__x_axis_units = 'bins'
_plot_clustering_report__y_to_x_aspect_ratio_override = 1.78
_plot_clustering_report__plot_window = [0, 2400]
_plot_clustering_report__cluster_stats_sv_data_var = 'Sv'
_plot_clustering_report__overlay_windows = windows

plot_clustering_report(
    ds_normalized=embed_results_ds_normalized,
    clustering_results=run_hdbscan_clustering_results,
    clustering_model=run_hdbscan_clustering_model,
    ds_Sv=rechunk_dive_mvbs_ds_Sv,
    dataset_name=_plot_clustering_report__dataset_name,
    ml_result_name=_plot_clustering_report__ml_result_name,
    x_axis_units=_plot_clustering_report__x_axis_units,
    y_to_x_aspect_ratio_override=_plot_clustering_report__y_to_x_aspect_ratio_override,
    plot_window=_plot_clustering_report__plot_window,
    cluster_stats_sv_data_var=_plot_clustering_report__cluster_stats_sv_data_var,
    overlay_windows=_plot_clustering_report__overlay_windows,
    overlay_fit_style={'color': '#23FFFF', 'linewidth': 3.0},
    overlay_bound_style={'color': '#D08BFF', 'linewidth': 1.8, 'linestyle': ':'},
)

*End of included section: processing_lvl_4.yaml*

## Included: processing_lvl_5_sv_codes.yaml

Steps: generate_sv_codes, sv_code_depth_table

### Step: `generate_sv_codes`
**Op:** `custom` (custom / unregistered)

Write per-dive SVCode CSVs from the embedded clustering.

**Parameters:**
- `windows` list
- `output_dir` path
- `dataset_name` str
- `ml_result_name` str
- `range_var` str
- `ping_time_bin` str

In [ ]:
# --- Parameters ---
_generate_sv_codes__windows = windows
_generate_sv_codes__output_dir = './outputs/sv_codes'
_generate_sv_codes__dataset_name = 'ml_dataset'
_generate_sv_codes__ml_result_name = 'hdbscan_results'
_generate_sv_codes__range_var = 'depth'
_generate_sv_codes__ping_time_bin = '10s'

_result = generate_sv_codes(
    ds=embed_results_ds_normalized,
    windows=_generate_sv_codes__windows,
    output_dir=_generate_sv_codes__output_dir,
    dataset_name=_generate_sv_codes__dataset_name,
    ml_result_name=_generate_sv_codes__ml_result_name,
    range_var=_generate_sv_codes__range_var,
    ping_time_bin=_generate_sv_codes__ping_time_bin,
)
code_csv_paths = _result['code_csv_paths']
summary_csv_path = _result['summary_csv_path']
dive_labels = _result['dive_labels']

### Step: `sv_code_depth_table`
**Op:** `custom` (custom / unregistered)

Percent of dive duration per depth interval per code.

**Parameters:**
- `summary_csv_path` path
- `output_dir` path
- `depth_interval_m` float

In [ ]:
# --- Parameters ---
_sv_code_depth_table__summary_csv_path = summary_csv_path
_sv_code_depth_table__output_dir = './outputs/sv_codes'
_sv_code_depth_table__depth_interval_m = 200.0

_result = sv_code_depth_table(
    summary_csv_path=_sv_code_depth_table__summary_csv_path,
    output_dir=_sv_code_depth_table__output_dir,
    depth_interval_m=_sv_code_depth_table__depth_interval_m,
)
per_dive_csv_path = _result['per_dive_csv_path']
combined_csv_path = _result['combined_csv_path']
totals_csv_path = _result['totals_csv_path']

*End of included section: processing_lvl_5_sv_codes.yaml*